In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import training_lib as tl
from helperfunctions import intern_constants as ic
from helperfunctions.pretty_print import PrettyPrint as pp
import numpy as np
import pandas as pd
import os
import torch.optim as optim
from torch import nn
import torch 

### Set config for training

In [ ]:
# cfg = hfn.TrainConfig(config_name="iter_1", depth=5, epochs=5)

# cfg = hfn.TrainConfig(config_name="iter_1", depth=5, epochs=5, n_restarts=5)
cfg = hfn.TrainConfig(config_name="layers", epochs=10, width_decay=0.5, n_restarts=1, patience=10, min_delta=0,lr=1e-3, weight_decay=1e-2, depth=5, choose_val_set=1)



In [ ]:
print(cfg.n_restarts)

### Make dataloaders

In [ ]:
#files = glob(os.path.join(PATH_IMPUTED, "*.csv") )


# train_loader, val_loader, test_loader = hfn.build_dataloaders(
#     train_csv_dir=PATH_PC_FILTERING,
#     val_csv_dir=PATH_IMPUTED,
#     test_csv_dir=PATH_IMPUTED,
#     cfg=cfg
# )

train_loader, val_loader, test_loader = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir=ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_IMPUTED,
    cfg=cfg
)

### Data time periods
Training time period is fixed.
To change it, go to helperfunctions.intern_constants

In [ ]:
print(f"train start{ic.TRAIN_START} - train end: {ic.TRAIN_END}\n"
      f"val start:{cfg.val_start_time} - val end: {cfg.val_end_time}\n"
      f"test start: {cfg.test_start_time} - test end: {cfg.test_end_time}")

## Data split overview

In [ ]:
def get_n(start:str, end:str):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    return ((end - start) // pd.Timedelta("10min")) +1

n_train = hfn.get_n_of_train_timestamps()
n_val = get_n(cfg.val_start_time,cfg.val_end_time)
n_test = get_n(cfg.test_start_time, cfg.test_end_time)

print(f"train unique ts: {n_train}\n"
      f"val unique ts: {n_val}\n"
      f"test unique ts:{n_test}\n")

print(f"total unique timestamps = {n_train + n_val+ n_test}\n"
      f"split unique train = {round(100/(n_train + n_val + n_test)*n_train,0)}%\n"
      f"split unique val = {round(100/(n_train + n_val + n_test)*n_val,0)}%\n"
      f"split unique test = {round(100/(n_train + n_val + n_test)*n_test,0)}%\n")


In [ ]:
print(f"available timestamps:\n" 
      f"train-loader: {len(train_loader.dataset)}\n"
      f"val_loader: {len(val_loader.dataset)}\n"
      f"test_loader: {len(test_loader.dataset)}\n")

In [ ]:
print(cfg.device)

### Number of input signals

In [ ]:
print(len(hfn.load_feature_order()))

### Training with early stopping

In [ ]:
n_restarts = cfg.n_restarts
base_seed = cfg.seed

best_val = np.inf
best_run = None
best_history = None
run_histories = []
filename_prefix = "pre_model_run"

for i in range(0, n_restarts):
    
    cfg.set_seed(cfg.seed_list[i])
    
    model = tl.Autoencoder(cfg=cfg).to(cfg.device)

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    es = tl.EarlyStopping(min_delta=cfg.min_delta, patience=cfg.patience)
    
    fn_pre = f"{filename_prefix}{i}"

    history_info, _ = tl.train_with_early_stopping(
        model= model,
        train_loader=train_loader,
        val_loader = val_loader,
        optimizer = optimizer,
        config=cfg,
        es=es,
        loss_fn=nn.MSELoss(reduction="none"),
        scaler=None,
        filename_prefix=fn_pre,
    )

    run_histories.append(history_info["history"])
    curr_best = float(history_info["best_val"])
    
    if curr_best < best_val:
        best_val = curr_best
        best_run = i
        #best_history = history_info

# Test loading the model
best_path = ic.PATH_TO_BEST_MODEL_DIR /f"{filename_prefix}{best_run}" /f"{filename_prefix}{best_run}.pth"
model_state = torch.load(best_path, map_location=cfg.device, weights_only=False)
# torch.save(model_state, PATH_TO_BEST_MODEL_DIR / f"{filename_prefix}{best_run}_{BEST_MODEL}")

print(f"Best run = {best_run} | best val RE = {best_val:.5f}"
      f"model at path: {best_path}")
#TODO safe the history
######################

### Test with same seed

In [ ]:

for _ in range(2):
    
    n_restarts = cfg.n_restarts
    base_seed = cfg.seed

    best_val = np.inf
    best_run = None
    best_history = None
    run_histories = []
    filename_prefix = "pre_model_run"
    for i in range(0, n_restarts):
        
        cfg.set_seed(cfg.seed_list[i])
        
        model = tl.Autoencoder(cfg=cfg).to(cfg.device)

        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        es = tl.EarlyStopping(min_delta=cfg.min_delta, patience=cfg.patience)
        
        fn_pre = f"{filename_prefix}{i}"

        history_info, _ = tl.train_with_early_stopping(
            model= model,
            train_loader=train_loader,
            val_loader = val_loader,
            optimizer = optimizer,
            config=cfg,
            es=es,
            loss_fn=nn.MSELoss(reduction="none"),
            scaler=None,
            filename_prefix=fn_pre,
        )

        run_histories.append(history_info["history"])
        curr_best = float(history_info["best_val"])
        
        if curr_best < best_val:
            best_val = curr_best
            best_run = i
            #best_history = history_info
    
        train_loader = hfn.rebuild_grouped_loader(train_loader,
                                                seed=cfg.seed, 
                                                shuffle=True, 
                                                batch_size=cfg.batch_size)
        # val_loader = hfn.rebuild_grouped_loader(val_loader,
        #                                         seed=cfg.seed,
        #                                         shuffle=False,
        #                                         batch_size=cfg.batch_size)
    
    pp.print_learning_curve(run_histories[best_run])
tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR)

### free Memory

In [ ]:

val_loader = None

In [ ]:
pp.print_learning_curve(run_histories[best_run])

#free memory
#run_histories = None

In [ ]:
ae_list = tl.get_model_results(src=ic.PATH_TO_BEST_MODEL_DIR, best_n=1)
model, _ , ckpt, _, _ = ae_list[0]
model.eval()


### Evaluation

In [ ]:
df_test_eval = tl.eval_model(model,
                             data_loader= test_loader,
                             device=cfg.device,
                             loss_fn=nn.MSELoss(reduction="none"),
                             )

df_test_eval.head(24)


### free memory

In [ ]:
#test_loader = None

### Check
Load model

In [ ]:
ae_list= tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR, best_n=1)
p = ae_list[0][2]['dir_path']

df_test_eval = pd.read_csv(os.path.join(p, ic.EVAL_FILE_NAME))

## Check REs

### Visualization of test set evaluation

In [ ]:
pp.print_loss(
            df_test_eval, 
            dpi=300,
            wt_id=[5],
            y_limits=((0,1),(1,35)), 
            )

In [ ]:
pp.print_loss(
            df_test_eval, 
            dpi=300, 
            y_limits=((0,0.2),(0.2,1)), 
            wt_id=[2]
            )

In [ ]:
df_test_eval.head()